# NB3 — Retrieval: complementary evidence, graph vs flat (Lens 3 / M3)

For each pre-registered query: a **flat** hybrid baseline (BM25 + kNN, RRF) vs a
**graph-expanded** result set (flat hits enriched with theme-sibling + shelf-
attached chunks, re-ranked), and the annotated **delta** (graph − flat).

Reloads the Layer A + Leiden-theme graph NB2 persisted. kNN uses real BGE-base
vectors (chunk cache + an in-space query embedding), not the dim-8 mock.

In [1]:
import sys, time
sys.path.insert(0, ".")
import cs_common as cs

OUT = cs.artifacts_dir("nb3_retrieval")
plt = cs.init_mpl()
K = cs.CONFIG["retrieval_k"]
print("artifacts ->", OUT, "| k =", K)

artifacts -> /mnt/workspaces/wisefood/foodscholar-lib/notebooks/case_study/artifacts/nb3_retrieval | k = 10


## 1. Reload Layer A + Leiden themes (from NB2)

In [2]:
fs = cs.get_fs(with_llm=False)
loaded = cs.load_graph(fs, themes=True, cards=False)
print("reloaded:", loaded)
anchors = cs.resolve_anchors(fs)
env = cs.capture_env(fs, extra={"reloaded": loaded, "retrieval_k": K,
                                "theme_build": "leiden (from NB2 shared snapshot)"})
cs.save_json(OUT / "env.json", env)
assert loaded["themes"] > 0, "no themes in snapshot — run NB2 first"

2026-06-24T11:47:33.697450Z [info     ] corpus.loaded                  config_hash=95894a60f0d5cd7b n=34359


reloaded: {'chunks': 34359, 'shelves': 1020, 'themes': 41, 'cards': 0}


## 2. Flat baseline — hybrid BM25 + kNN (RRF)

`InMemoryChunkStore.search` is itself a BM25 + kNN RRF hybrid. For a query-side
embedding in the same BGE-base space as the cached chunk vectors we encode the
query with the real HF embedder (`cs.bge_query_vector`) and fuse the store's
hybrid hits with a pure-kNN list (RRF) so the ranking is genuinely semantic.

In [3]:
def rrf(*ranked_lists, k=60):
    """Reciprocal-rank fusion over several ranked id lists -> {id: score}."""
    scores = {}
    for lst in ranked_lists:
        for rank, cid in enumerate(lst):
            scores[cid] = scores.get(cid, 0.0) + 1.0 / (k + rank + 1)
    return scores

def flat_retrieve(query, k=K):
    """Hybrid flat retrieval: store BM25+kNN hybrid fused with pure BGE kNN."""
    hybrid = fs.chunk_store.search(query, k=k * 3)            # BM25+kNN (mock-safe)
    qv = cs.bge_query_vector(query)                           # real BGE query vector
    knn = fs.chunk_store.knn_search_chunks(qv, k=k * 3)       # real semantic kNN
    fused = rrf([c.chunk_id for c in hybrid], [cid for cid, _ in knn])
    ranked = sorted(fused, key=lambda c: fused[c], reverse=True)[:k]
    return ranked, {cid: fused[cid] for cid in ranked}

flat = {}
for key, q in cs.CONFIG["queries"].items():
    ids, sc = flat_retrieve(q)
    flat[key] = {"query": q, "ids": ids, "scores": sc}
    print(f"[{key}] flat top-{K}: {len(ids)} hits; top score {max(sc.values()):.4f}")

/mnt/miniconda3/envs/foodscholar/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


No device provided, using cpu


Loading SentenceTransformer model from BAAI/bge-base-en-v1.5.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 10.97it/s]

...

Batches: 100%|██████████| 1/1 [00:00<00:00, 10.01it/s]

[fish] flat top-10: 10 hits; top score 0.0311


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 12.15it/s]

[dietary_fibre] flat top-10: 10 hits; top score 0.0283


## 3. Graph expansion

From the flat top hits, gather their `theme_ids` (+ the anchor `shelf_id`), pull
sibling chunks from those themes / the shelf, merge with the flat hits, and
re-rank by `flat_score + theme-membership weight`.

In [4]:
THEME_WEIGHT = 0.5

def graph_expand(key, k=K):
    base_ids = flat[key]["ids"]
    base_scores = dict(flat[key]["scores"])
    # themes activated by the flat hits + the anchor shelf's themes
    activated_themes = set()
    for cid in base_ids:
        c = fs.chunk_store.get(cid)
        if c is not None:
            activated_themes.update(c.theme_ids)
    anchor_shelf = anchors[key]["shelf"]
    for th in anchor_shelf.themes():
        activated_themes.add(th.theme_id)
    # sibling chunks from those themes
    sibling_ids = set()
    for tid in activated_themes:
        sibling_ids.update(fs.graph_store.get_chunks_for_theme(tid))
    # also shelf-attached chunks (subtree anchor membership)
    shelf_chunk_ids = {c.chunk_id for c in anchor_shelf.chunks()}
    candidate_ids = set(base_ids) | sibling_ids | shelf_chunk_ids
    # re-rank: flat score (if present) + theme-membership weight
    qv = cs.bge_query_vector(flat[key]["query"])
    knn = dict(fs.chunk_store.knn_search_chunks(qv, k=len(candidate_ids) + 5,
                                                candidate_ids=list(candidate_ids)))
    final = {}
    for cid in candidate_ids:
        s = base_scores.get(cid, 0.0) + knn.get(cid, 0.0)
        c = fs.chunk_store.get(cid)
        if c is not None and activated_themes.intersection(c.theme_ids):
            s += THEME_WEIGHT * knn.get(cid, 0.0)
        final[cid] = s
    ranked = sorted(final, key=lambda c: final[c], reverse=True)[:k]
    return ranked, {cid: final[cid] for cid in ranked}, sorted(activated_themes)

graph = {}
for key in cs.CONFIG["queries"]:
    ids, sc, themes = graph_expand(key)
    graph[key] = {"ids": ids, "scores": sc, "activated_themes": themes}
    print(f"[{key}] graph top-{K}: {len(ids)} hits; {len(themes)} themes activated")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 12.11it/s]

[olive_oil] graph top-10: 10 hits; 6 themes activated


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 13.84it/s]

...

[fish] graph top-10: 10 hits; 16 themes activated


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  7.02it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  6.96it/s]

[dietary_fibre] graph top-10: 10 hits; 14 themes activated


## 4. Delta (graph − flat), annotated

Reason per delta chunk: `different-document` (source doc not in the flat set) /
`different-food-entity` (FoodOn ids disjoint from the flat set) / `paraphrase`
(same doc + overlapping entities). A heuristic the author confirms in prose.

In [5]:
def delta_reason(cid, flat_docs, flat_foodon):
    c = fs.chunk_store.get(cid)
    if c is None:
        return "unresolved"
    if c.source_doc_id not in flat_docs:
        return "different-document"
    if set(c.foodon_ids).isdisjoint(flat_foodon):
        return "different-food-entity"
    return "paraphrase"

deltas = {}
for key in cs.CONFIG["queries"]:
    flat_ids = flat[key]["ids"]; graph_ids = graph[key]["ids"]
    flat_docs = {fs.chunk_store.get(c).source_doc_id for c in flat_ids if fs.chunk_store.get(c)}
    flat_foodon = set()
    for c in flat_ids:
        ch = fs.chunk_store.get(c)
        if ch: flat_foodon.update(ch.foodon_ids)
    delta_rows = []
    for cid in graph_ids:
        if cid in set(flat_ids):
            continue
        c = fs.chunk_store.get(cid)
        # which activated theme does this chunk belong to?
        tid = next((t for t in (c.theme_ids if c else []) if t in graph[key]["activated_themes"]), None)
        delta_rows.append({
            "chunk_id": cid,
            "source_doc": c.source_doc_id if c else None,
            "excerpt": cs.excerpt(c.text, 240) if c else None,
            "theme_id": tid,
            "reason": delta_reason(cid, flat_docs, flat_foodon),
        })
    deltas[key] = {
        "anchor": key, "query": cs.CONFIG["queries"][key],
        "flat_ids": flat_ids, "graph_ids": graph_ids, "delta": delta_rows,
    }
    cs.save_json(OUT / f"{key}_flat.json", {"anchor": key, "query": cs.CONFIG["queries"][key],
                                            "ids": flat_ids,
                                            "hits": [{"chunk_id": c, "excerpt": cs.excerpt(fs.chunk_store.get(c).text, 180)} for c in flat_ids if fs.chunk_store.get(c)]})
    cs.save_json(OUT / f"{key}_graph.json", {"anchor": key, "ids": graph_ids,
                                             "activated_themes": graph[key]["activated_themes"]})
    cs.save_json(OUT / f"{key}_delta.json", deltas[key])
    from collections import Counter
    rc = Counter(r["reason"] for r in delta_rows)
    print(f"[{key}] delta = {len(delta_rows)} new chunks; reasons {dict(rc)}")

[olive_oil] delta = 5 new chunks; reasons {'paraphrase': 2, 'different-document': 3}
[legume] delta = 5 new chunks; reasons {'different-document': 5}
[fish] delta = 3 new chunks; reasons {'paraphrase': 1, 'different-document': 2}
[dietary_fibre] delta = 7 new chunks; reasons {'different-document': 4, 'different-food-entity': 1, 'paraphrase': 2}


## 5. Comparison figures (flat | graph-expanded, delta highlighted)

In [ ]:
REASON_COLOR = {
    "different-document": cs.COLORS["amber"],
    "different-food-entity": cs.COLORS["purple"],
    "paraphrase": cs.COLORS["teal"],
    "unresolved": cs.COLORS["grey"],
}

def compare_fig(key):
    flat_ids = flat[key]["ids"]; graph_ids = graph[key]["ids"]
    flat_set = set(flat_ids)
    reason_by_id = {r["chunk_id"]: r["reason"] for r in deltas[key]["delta"]}
    n_delta = len(deltas[key]["delta"])
    fig, ax = cs.slide(
        f"Flat vs graph-expanded retrieval — {key}",
        eyebrow="lens 3 · M3",
        caption=f'Query: "{cs.CONFIG["queries"][key]}"',
    )
    cs.kpi(ax, 0.82, 0.99, f"{n_delta}/{cs.CONFIG['retrieval_k']}", "graph-only hits",
           color=cs.COLORS["purple"], w=0.17, h=0.24, value_fontsize=24)
    def column(ids, x0, head, color, mark_delta):
        cs.chip(ax, x0, 0.96, head, color=color, fontsize=14)
        top = 0.84
        step = 0.083
        for i, cid in enumerate(ids):
            c = fs.chunk_store.get(cid)
            y = top - i * step
            is_delta = mark_delta and cid not in flat_set
            ax.text(x0, y, f"{i+1:2d}.", fontsize=11, weight="bold",
                    color=cs.COLORS["muted"], va="top", family="DejaVu Sans Mono")
            # delta rows shorten the excerpt to leave room for an inline reason
            # chip on the same line — drawing the chip below the row collided with
            # the next row whenever a column had many delta hits (e.g. dietary_fibre 7/10).
            elen = 44 if is_delta else 58
            ax.text(x0 + 0.028, y, cs.excerpt(c.text, elen) if c else cid,
                    fontsize=10.5, color=cs.COLORS["ink"], va="top")
            if is_delta:
                rsn = reason_by_id.get(cid, "")
                cs.chip(ax, x0 + 0.345, y - 0.012, rsn,
                        color=REASON_COLOR.get(rsn, cs.COLORS["grey"]),
                        fontsize=8.5, weight="normal", pad=0.25)
    column(flat_ids, 0.0, "flat — BM25 + kNN (RRF)", cs.COLORS["teal"], mark_delta=False)
    column(graph_ids, 0.50, "graph-expanded", cs.COLORS["purple"], mark_delta=True)
    cs.save_slide(fig, OUT / f"{key}_compare.png")
    print("wrote", OUT / f"{key}_compare.png")

for key in cs.CONFIG["queries"]:
    compare_fig(key)

## 6. summary.md

In [7]:
from collections import Counter
lines = ["# NB3 — Retrieval flat vs graph (M3) — summary", "",
         f"- Build/backend: in-memory, Leiden themes from NB2 snapshot; k={K}.",
         "- Flat = `InMemoryChunkStore.search` (BM25+kNN RRF) fused with real BGE-base "
         "kNN. Graph = flat ∪ theme-siblings ∪ anchor-shelf chunks, re-ranked with a "
         f"theme-membership weight ({THEME_WEIGHT}).", "", "## Deltas"]
for key in cs.CONFIG["queries"]:
    d = deltas[key]["delta"]
    rc = Counter(r["reason"] for r in d)
    lines.append(f"- **{key}**: {len(d)} graph-only chunks of {K}; reasons {dict(rc)}.")
    if not d:
        lines.append("  - *Empty/near-empty delta reported as a limitation* — the flat "
                     "hybrid already saturates this query's relevant chunks.")
lines += ["", "## Files", "env.json, {anchor}_flat.json, {anchor}_graph.json, "
          "{anchor}_delta.json, {anchor}_compare.png.", "",
          "## Deviations / limitations",
          "- BM25 is the in-memory store's token-overlap approximation (no ES); the kNN "
          "side uses real BGE-base vectors (chunk cache + in-space query embedding).",
          "- The delta `reason` is a heuristic (doc / FoodOn-entity / paraphrase); the "
          "narrative confirms a sample by hand.",
          "", "## Acceptance",
          "- [x] both queries produce flat + graph result sets",
          "- [x] delta computed and each delta hit annotated",
          "- [x] empty/noisy deltas reported, not hidden"]
(OUT / "summary.md").write_text("\n".join(lines))
print("\n".join(lines))

# NB3 — Retrieval flat vs graph (M3) — summary

- Build/backend: in-memory, Leiden themes from NB2 snapshot; k=10.
- Flat = `InMemoryChunkStore.search` (BM25+kNN RRF) fused with real BGE-base kNN. Graph = flat ∪ theme-siblings ∪ anchor-shelf chunks, re-ranked with a theme-membership weight (0.5).

## Deltas
- **olive_oil**: 5 graph-only chunks of 10; reasons {'paraphrase': 2, 'different-document': 3}.
- **legume**: 5 graph-only chunks of 10; reasons {'different-document': 5}.
- **fish**: 3 graph-only chunks of 10; reasons {'paraphrase': 1, 'different-document': 2}.
- **dietary_fibre**: 7 graph-only chunks of 10; reasons {'different-document': 4, 'different-food-entity': 1, 'paraphrase': 2}.

## Files
env.json, {anchor}_flat.json, {anchor}_graph.json, {anchor}_delta.json, {anchor}_compare.png.

## Deviations / limitations
- BM25 is the in-memory store's token-overlap approximation (no ES); the kNN side uses real BGE-base vectors (chunk cache + in-space query embedding).
- The delta `rea